# Advanced Problems with Solutions: Sorting Iterables in Python

This notebook expands the topic of sorting iterables with `sorted()`.

The original idea is simple: `sorted()` works with **any iterable**, not only sequences.  
The advanced part is understanding what happens with custom iterables, one-shot iterators, key functions, stability, memory use, partial sorting, grouping, and performance.

## Learning goals

By the end, you should be able to:

- Sort any iterable, including custom iterable classes
- Explain why `sorted()` always returns a new `list`
- Explain why sorting an iterator consumes it
- Use `key=`, `reverse=`, tuple keys, and stable sorting correctly
- Build reusable custom iterables that work well with `sorted()`
- Avoid common mistakes with randomness, global state, and one-shot iterators
- Use `heapq` for top-k problems instead of sorting everything
- Sort records safely when values can be missing
- Sort custom objects with dataclasses and comparison methods
- Combine sorting with `itertools.groupby`


## Setup

Everything in this notebook is self-contained.


In [1]:
from __future__ import annotations

from collections import Counter
from dataclasses import dataclass, field
from functools import cmp_to_key
from heapq import nlargest, nsmallest
from itertools import groupby, islice
from operator import attrgetter, itemgetter
from random import Random
from typing import Any, Callable, Iterable, Iterator, TypeVar

T = TypeVar("T")


# Section 1 — Warm-up: `sorted()` works with any iterable

`sorted(iterable)` accepts any iterable and returns a **new list**.


In [2]:
print(sorted([5, 3, 1, 4, 2]))
print(sorted((5, 3, 1, 4, 2)))
print(sorted({5, 3, 1, 4, 2}))
print(sorted("python"))

result = sorted(range(10, 0, -2))
print(result)
print(type(result))


[1, 2, 3, 4, 5]
[1, 2, 3, 4, 5]
[1, 2, 3, 4, 5]
['h', 'n', 'o', 'p', 't', 'y']
[2, 4, 6, 8, 10]
<class 'list'>


# Problem 1 — Sort a generator expression

Create a generator expression that produces squares from `10` down to `1`, then sort the generated values.

Questions:

1. What type is the generator?
2. What type is the sorted result?
3. Can you reuse the generator after sorting?


In [3]:
squares = (x * x for x in range(10, 0, -1))

print("generator object:", squares)
print("has __iter__:", hasattr(squares, "__iter__"))
print("has __next__:", hasattr(squares, "__next__"))

sorted_squares = sorted(squares)

print("sorted result:", sorted_squares)
print("sorted result type:", type(sorted_squares))

# The generator has been consumed by sorted().
print("trying to list the generator again:", list(squares))

assert sorted_squares == [1, 4, 9, 16, 25, 36, 49, 64, 81, 100]
assert list(squares) == []


generator object: <generator object <genexpr> at 0x000002201530E9B0>
has __iter__: True
has __next__: True
sorted result: [1, 4, 9, 16, 25, 36, 49, 64, 81, 100]
sorted result type: <class 'list'>
trying to list the generator again: []


## Key idea

Sorting requires seeing all values before it can know the final order.  
So `sorted()` must consume the input iterable and materialize the result as a list.


# Problem 2 — Build a better random integer iterable

The lesson example used a custom `RandomInts` iterable.

A common mistake is to call `random.seed(...)` inside an iterator. That mutates the global random generator and can affect other code.

Task:

Create `RandomInts` so that:

- It is reusable
- Each new pass produces the same values for the same seed
- It does not mutate global random state
- It supports `len()`
- It can be sorted


In [4]:
class RandomInts:
    def __init__(self, length: int, *, seed: int = 0, lower: int = 0, upper: int = 10) -> None:
        if length < 0:
            raise ValueError("length must be non-negative")
        if lower > upper:
            raise ValueError("lower must be <= upper")

        self.length = length
        self.seed = seed
        self.lower = lower
        self.upper = upper

    def __len__(self) -> int:
        return self.length

    def __iter__(self) -> Iterator[int]:
        rng = Random(self.seed)

        for _ in range(self.length):
            yield rng.randint(self.lower, self.upper)


randoms = RandomInts(10, seed=0, lower=0, upper=10)

first_pass = list(randoms)
second_pass = list(randoms)
sorted_pass = sorted(randoms)

print("first pass: ", first_pass)
print("second pass:", second_pass)
print("sorted:     ", sorted_pass)

assert first_pass == second_pass
assert sorted_pass == sorted(first_pass)
assert len(randoms) == 10


first pass:  [6, 6, 0, 4, 8, 7, 6, 4, 7, 5]
second pass: [6, 6, 0, 4, 8, 7, 6, 4, 7, 5]
sorted:      [0, 4, 4, 5, 6, 6, 6, 7, 7, 8]


# Problem 3 — Show that `sorted()` consumes one-shot iterators

Create a `RandomIterator` that is intentionally single-use. Sort it once, then try to sort it again.


In [5]:
class RandomIterator:
    def __init__(self, length: int, *, seed: int = 0, lower: int = 0, upper: int = 10) -> None:
        self.remaining = length
        self.rng = Random(seed)
        self.lower = lower
        self.upper = upper

    def __iter__(self) -> "RandomIterator":
        return self

    def __next__(self) -> int:
        if self.remaining <= 0:
            raise StopIteration

        self.remaining -= 1
        return self.rng.randint(self.lower, self.upper)


it = RandomIterator(10, seed=0)

print("first sorted call:", sorted(it))
print("second sorted call:", sorted(it))

# The second call is empty because the iterator was exhausted.


first sorted call: [0, 4, 4, 5, 6, 6, 6, 7, 7, 8]
second sorted call: []


# Problem 4 — Write a sorting diagnostic helper

Write `sort_preview(iterable, n=5)` that returns:

- the first `n` sorted values
- the total number of sorted values

This function may materialize the iterable because sorting already materializes it.


In [6]:
def sort_preview(iterable: Iterable[T], n: int = 5) -> dict[str, Any]:
    if n < 0:
        raise ValueError("n must be non-negative")

    values = sorted(iterable)
    return {
        "preview": values[:n],
        "count": len(values),
    }


print(sort_preview(RandomInts(20, seed=42)))
print(sort_preview((x % 7 for x in range(30)), n=10))

assert sort_preview([3, 1, 2]) == {"preview": [1, 2, 3], "count": 3}


{'preview': [0, 0, 0, 1, 1], 'count': 20}
{'preview': [0, 0, 0, 0, 0, 1, 1, 1, 1, 1], 'count': 30}


# Section 2 — Sorting with `key=`

`key=` is usually the most important sorting argument.

Instead of comparing entire objects directly, Python computes a key for each item and sorts by that key.


In [7]:
words = ["pear", "fig", "apple", "kiwi", "banana"]

print("alphabetical:", sorted(words))
print("by length:", sorted(words, key=len))
print("by last letter:", sorted(words, key=lambda word: word[-1]))
print("by length then alphabetical:", sorted(words, key=lambda word: (len(word), word)))


alphabetical: ['apple', 'banana', 'fig', 'kiwi', 'pear']
by length: ['fig', 'pear', 'kiwi', 'apple', 'banana']
by last letter: ['banana', 'apple', 'fig', 'kiwi', 'pear']
by length then alphabetical: ['fig', 'kiwi', 'pear', 'apple', 'banana']
